# ACTIS — Full Validation Notebook
Runs all 12 phases. Set runtime to **GPU (T4)**.

## Phase 0 — Setup

In [ ]:
!git clone https://github.com/veerasagar/ACTIS-MCE442P.git
%cd MCE442P

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bert-score sentence-transformers

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || print('Upload kaggle.json first')
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p data/nf-cse-cic-ids2018-v2 data/nf-bot-iot-v2
!kaggle datasets download -d dhoogla/nfcsecicids2018v2 -p data/nf-cse-cic-ids2018-v2 --unzip -q
!kaggle datasets download -d dhoogla/nfbotiotv2 -p data/nf-bot-iot-v2 --unzip -q
print('Datasets downloaded')

## Phase 1 — Data Pipeline + Feature Engineering

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from src.data.loader import FedIntelDataLoader
from src.data.feature_engineer import FeatureEngineer
import numpy as np

loader = FedIntelDataLoader()
data = loader.load_and_partition()
for cid in sorted(data.keys()):
    d = data[cid]
    print(f'Node {cid}: {len(d["data"]):>11,} rows  ({d["dataset"]})')

fe = FeatureEngineer()
X = np.random.rand(100, 41).astype('float32')
print(f'\nFeatures: {X.shape} → {fe.transform(X).shape} ({fe.N_DERIVED} derived)')

## Phase 2 — Local IDS (No Federation)

In [ ]:
from src.data.preprocessor import Preprocessor
from src.local_node.ids_model import IDSModel
from src.local_node.ids_trainer import IDSTrainer

all_labels = set()
for c in data.values(): all_labels.update(c['data'][c['label_col']].unique())
unified = sorted(all_labels)

for cid in sorted(data.keys()):
    p = Preprocessor(); p.set_unified_labels(unified)
    X_tr, X_te, y_tr, y_te = p.prepare(data[cid], max_samples=20000)
    model = IDSModel(input_dim=41, num_classes=p.num_classes)
    t = IDSTrainer(model=model)
    t.train(X_tr, y_tr, epochs=10, verbose=False)
    m = t.evaluate(X_te, y_te)
    ds = 'IDS2018' if cid in ['A','B'] else 'BoT-IoT'
    print(f'Node {cid} ({ds}): Acc={m["accuracy"]*100:.2f}%  F1={m["f1"]:.4f}')

## Phase 3 — Federated Learning (4 Nodes)

In [ ]:
from src.server.fl_simulation import FLSimulation

sim = FLSimulation(max_samples=30000)
results = sim.run(num_rounds=10)

for cid in sorted(results['final_eval'].keys()):
    r = results['final_eval'][cid]
    ds = 'IDS2018' if cid in ['A','B'] else 'BoT-IoT'
    print(f'Node {cid} ({ds}): Acc={r["accuracy"]*100:.2f}%  F1={r["f1"]:.4f}  P={r["precision"]:.4f}  R={r["recall"]:.4f}')
print(f'Loss: {results["round_metrics"][0]["avg_loss"]:.4f} → {results["round_metrics"][-1]["avg_loss"]:.4f}')

## Phase 4 — Privacy (PII + DP)

In [ ]:
import torch
from src.local_node.node import PrivacyNode

node = PrivacyNode(company_id='A')
flow = {'PROTOCOL': 6, 'L4_SRC_PORT': 12345, 'L4_DST_PORT': 80,
        'IN_BYTES': 500000, 'OUT_BYTES': 200, 'IN_PKTS': 1000,
        'OUT_PKTS': 5, 'FLOW_DURATION_MILLISECONDS': 100, 'TCP_FLAGS': 2}
result = node.process_flow(flow, 'ddos')
print(f'PII sanitized + shared: {bool(result)}')
node.print_summary()

# DP noise verification
model = IDSModel(input_dim=41, num_classes=9)
X = torch.randn(10, 41)
model.train()
diff = (model(X) - model(X)).abs().mean().item()
print(f'DP noise (train): {diff:.6f} (active={diff>0})')
model.eval()
diff = (model(X) - model(X)).abs().mean().item()
print(f'DP noise (eval):  {diff:.6f} (deterministic={diff==0})')

## Phase 5 — Zero-Day DDoS Detection

In [ ]:
import gc, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from src.local_node.zeroday_detector import ZeroDayDetector

df = pd.read_parquet('data/nf-bot-iot-v2/NF-BoT-IoT-V2.parquet')
bot_map = {'DDoS':'ddos','DoS':'dos','Reconnaissance':'reconnaissance','Benign':'benign','Theft':'theft'}
df['u'] = df['Attack'].map(bot_map).fillna('benign')
classes = sorted(df['u'].unique()); lmap = {n:i for i,n in enumerate(classes)}
chunks = [df[df['u']==c].sample(n=min(len(df[df['u']==c]),3000), random_state=42) for c in classes]
df_s = pd.concat(chunks).sample(frac=1, random_state=42); del df; gc.collect()
fc = [c for c in df_s.columns if c not in ['Attack','Label','u'] and pd.api.types.is_numeric_dtype(df_s[c])][:41]
X = fe.transform(df_s[fc].replace([np.inf,-np.inf],0).fillna(0).values.astype(np.float32))
y = np.array([lmap[l] for l in df_s['u']])
X = StandardScaler().fit_transform(X).astype(np.float32)
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
di = classes.index('ddos'); ki = [i for i in range(len(classes)) if i!=di]
Xt,yt = X_tr[y_tr!=di], y_tr[y_tr!=di]
m = IDSModel(input_dim=X.shape[1], num_classes=len(classes))
t = IDSTrainer(model=m); t.train(Xt,yt,epochs=15,verbose=False)
for a,p in [(0.3,90),(0.5,90),(0.3,95)]:
    d = ZeroDayDetector(m,alpha=a,percentile=p); d.fit_thresholds(Xt,yt)
    r = d.detect_and_report(X_te,y_true=y_te,known_class_idx=ki)
    print(f'α={a} p={p}: {r.get("unknown_flagged",0)}/{r.get("unknown_count",0)} DDoS  Rate={r.get("unknown_detection_rate",0)*100:.1f}%  FAR={r.get("known_false_alarm_rate",0)*100:.1f}%')

## Phase 6 — RAG + Cross-Org Intelligence

In [ ]:
import shutil
from src.server.global_kb import GlobalKnowledgeBase
from src.server.aggregator import Aggregator
from src.server.rag_engine import RAGEngine
from src.server.immunity import ImmunityEngine

kb = GlobalKnowledgeBase(persist_dir='/tmp/actis_kb')
kb.populate_mitre()
rag = RAGEngine(kb, abstention_threshold=0.30, use_reranker=True)

for q, exp in [('DDoS amplification UDP', False), ('cooking recipes', True)]:
    r = rag.query(q)
    ok = '✓' if r['abstained']==exp else '✗'
    print(f'{ok} "{q}" → abstained={r["abstained"]} score={r["best_score"]:.3f}')

agg = Aggregator(kb)
agg.ingest({'analysis': {'attack_type':'ddos','attack_description':'UDP flood port 53',
    'mitre_technique_id':'T1498','mitre_tactic':'Impact','severity':'CRITICAL',
    'company_id':'A','recommended_defense':'Rate limit UDP 53'}})
hits = rag.find_similar_attacks('ddos','B')
imm = ImmunityEngine(rag)
rule = imm.generate_rules('ddos','B')
print(f'Cross-org threats A→B: {len(hits)}')
print(f'Firewall rule: {rule["rule"][:80]}...')
shutil.rmtree('/tmp/actis_kb', ignore_errors=True)

## Phase 7 — Report Quality (BERTScore)

In [ ]:
from src.evaluation.evaluate import _compute_bertscore
from src.data.summarizer import ThreatSummarizer

preds = ['DDoS attack detected with high-volume UDP flood targeting availability. MITRE T1498.']
refs = ['Distributed Denial of Service with volumetric UDP flood impacting availability.']
bs = _compute_bertscore(preds, refs)
print(f'BERTScore: P={bs["precision"]:.4f}  R={bs["recall"]:.4f}  F1={bs["f1"]:.4f}')
print(f'CyberRAG: F1=0.9400  |  ACTIS: F1={bs["f1"]:.4f}')

s = ThreatSummarizer(use_llm=True)
r = s.summarize_flow({'PROTOCOL':17,'L4_DST_PORT':53,'IN_BYTES':5000000,'OUT_BYTES':200,
    'IN_PKTS':10000,'OUT_PKTS':5,'FLOW_DURATION_MILLISECONDS':100,'TCP_FLAGS':0}, 'ddos', 'A')
print(f'Severity: {r["severity"]}  MITRE: {r["mitre_technique_id"]}  LLM: {r["llm_enhanced"]}')

## Phase 8 — ReGAIN Benchmark

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def run_binary(df, fc, name, ra, rp, rr):
    pos = df[df['is_attack']].sample(n=min(8000,df['is_attack'].sum()),random_state=42)
    neg = df[~df['is_attack']].sample(n=min(8000,(~df['is_attack']).sum()),random_state=42)
    sub = pd.concat([pos,neg])
    X = sub[fc].replace([np.inf,-np.inf],0).fillna(0).values.astype(np.float32)
    for i in range(X.shape[1]):
        u = np.percentile(X[:,i],99.9)
        if u>0: X[:,i]=np.clip(X[:,i],0,u)
    X = np.nan_to_num(fe.transform(X),nan=0,posinf=1e6,neginf=0)
    y = sub['is_attack'].astype(int).values
    X = StandardScaler().fit_transform(X).astype(np.float32)
    Xr,Xe,yr,ye = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    mdl = IDSModel(input_dim=X.shape[1],num_classes=2)
    tr = IDSTrainer(model=mdl); tr.train(Xr,yr,epochs=15,verbose=False)
    ev = tr.evaluate(Xe,ye)
    mdl.eval()
    with torch.no_grad(): pr = mdl(torch.tensor(Xe)).argmax(dim=1).numpy()
    p=precision_score(ye,pr,zero_division=0)*100; r=recall_score(ye,pr,zero_division=0)*100
    acc=ev['accuracy']*100
    v = '✅ WINS' if acc>float(ra) else '≈ PARITY'
    print(f'{name}: ACTIS={acc:.2f}% P={p:.1f}% R={r:.1f}% | ReGAIN={ra}% P≈{rp}% R≈{rr}% | {v}')

df18 = pd.read_parquet('data/nf-cse-cic-ids2018-v2/NF-CSE-CIC-IDS2018-V2.parquet')
f18 = [c for c in df18.columns if c not in ['Attack','Label'] and pd.api.types.is_numeric_dtype(df18[c])][:41]
df18['is_attack'] = df18['Attack'].isin(['DoS attacks-Hulk','DoS attacks-GoldenEye'])
run_binary(df18,f18,'TCP SYN Flood','98.82','91.0','98.6')
del df18; gc.collect()

dfb = pd.read_parquet('data/nf-bot-iot-v2/NF-BoT-IoT-V2.parquet')
fb = [c for c in dfb.columns if c not in ['Attack','Label'] and pd.api.types.is_numeric_dtype(dfb[c])][:41]
dfb['is_attack'] = dfb['Attack']=='DDoS'
run_binary(dfb,fb,'ICMP/UDP Flood','95.95','74.5','100')
del dfb; gc.collect()

## Phase 9 — Full Evaluation Suite

In [ ]:
!python3 -m src.evaluation.evaluate

## Phase 10 — Agent Analyzer + Sanitizer + PII Validator

In [ ]:
from src.local_node.agent_analyzer import AgentAnalyzer
from src.local_node.agent_sanitizer import AgentSanitizer
from src.local_node.pii_validator import PIIValidator

flow = {'PROTOCOL':17,'L4_SRC_PORT':1234,'L4_DST_PORT':53,'IN_BYTES':5000000,
        'OUT_BYTES':200,'IN_PKTS':10000,'OUT_PKTS':5,'FLOW_DURATION_MILLISECONDS':100,'TCP_FLAGS':0}

# Agent Analyzer
analyzer = AgentAnalyzer()
analysis = analyzer.analyze(flow, 'ddos', 'A')
print(f'Analyzer: {analysis["source"]} | {analysis["attack_type"]} | {analysis["mitre_technique_id"]} | {analysis["severity"]}')
print(f'Defense: {analysis["recommended_defense"]}')

# Agent Sanitizer
sanitizer = AgentSanitizer()
dirty = dict(analysis)
dirty['attack_description'] = 'DDoS from 192.168.1.100 to admin@corp.com'
sanitized = sanitizer.sanitize(dirty)
print(f'\nSanitizer:')
print(f'  Before: {dirty["attack_description"]}')
print(f'  After:  {sanitized["attack_description"]}')

# PII Validator
validator = PIIValidator()
clean, findings = validator.validate(sanitized)
print(f'\nPII Validator: clean={clean} findings={len(findings)}')
dirty_test = {'desc': 'Attack from 10.0.0.5 user john@evil.com'}
clean2, findings2 = validator.validate(dirty_test)
print(f'Dirty test: clean={clean2}')
for f in findings2: print(f'  {f["pii_type"]}: {f["match"]}')
print(f'Leakage rate: {validator.get_stats()["leakage_rate"]:.1f}%')

## Phase 11 — MITRE ATT&CK Mapper

In [ ]:
from src.data.mitre_mapper import MITREMapper

mapper = MITREMapper()
attacks = ['ddos','dos','brute_force','web_attack','botnet','infiltration','reconnaissance','theft']
print(f'{"Attack":<18s} {"Technique":<10s} {"Tactic":<20s} Severity')
print('-'*65)
for atk in attacks:
    m = mapper.map(atk)
    print(f'{atk:<18s} {m["technique_id"]:<10s} {m["tactic"]:<20s} {m["severity"]}')

## Phase 12 — Live Monitor Demo

In [ ]:
from src.live_monitor import LiveMonitor

monitor = LiveMonitor(company_id='A')
test_flows = [
    {'PROTOCOL':17,'L4_SRC_PORT':1234,'L4_DST_PORT':53,'IN_BYTES':5000000,'OUT_BYTES':200,
     'IN_PKTS':10000,'OUT_PKTS':5,'FLOW_DURATION_MILLISECONDS':100,'TCP_FLAGS':0},
    {'PROTOCOL':6,'L4_SRC_PORT':4444,'L4_DST_PORT':22,'IN_BYTES':500,'OUT_BYTES':500,
     'IN_PKTS':20,'OUT_PKTS':20,'FLOW_DURATION_MILLISECONDS':5000,'TCP_FLAGS':27},
    {'PROTOCOL':6,'L4_SRC_PORT':8080,'L4_DST_PORT':80,'IN_BYTES':100,'OUT_BYTES':50000,
     'IN_PKTS':2,'OUT_PKTS':100,'FLOW_DURATION_MILLISECONDS':200,'TCP_FLAGS':219},
]
for i, f in enumerate(test_flows):
    alert = monitor.process_flow(f)
    if alert:
        print(f'Flow {i+1}: [{alert["severity"]}] {alert["predicted_attack"].upper()} '
              f'conf={alert["confidence"]:.0%} MITRE={alert["mitre"]} zeroday={alert["is_zero_day"]}')
    else:
        print(f'Flow {i+1}: Benign')
monitor._print_summary()